In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../../scripts')

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
DATA_ROOT = '/data/a330d' #os.environ.get("DATA_ROOT", ".")
import numpy as np
import glob
import numpy as np
import json
import matplotlib as mpl
#mpl.rcParams["font.family"] = "monospace"

from plotting import plot_model_comparison

/data/a330d/miniforge3/envs/cellina-graph/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset_name = "merfish" # Options: merfish, crc

In [5]:
corr_dir = os.path.join(DATA_ROOT, f"datasets/{dataset_name}/correlations")
pattern = os.path.join(corr_dir, "*.json")
files = sorted(glob.glob(pattern))

rows = []
for fp in files:
    name = os.path.basename(fp)
    core = name[len("crc_"):-len(".json")] if dataset_name == "crc" else name[:-len(".json")]
    parts = core.split("_")
    sid = parts[0]
    model_name = parts[1]
    holdout_celltype = "_".join(parts[2:])
    try:
        with open(fp, "r") as f:
            data = json.load(f)
    except Exception:
        # skip unreadable/invalid json
        continue

    try:
        rows.append({
            "sid": f"crc_{sid}" if dataset_name == "crc" else sid,
            "model_name": model_name,
            "holdout_celltype": holdout_celltype,
            "n_deg": data.get("n_deg"),
            "spearman": data.get("spearman"),
            "pearson": data.get("pearson"),
            "precision": data.get("precision"),
            "direction_match": data.get("direction_match"),
            "direction_match_k": data.get("direction_match_k"),
            "direction_match_gt": data.get("direction_match_gt"),
            "mixing_index": data.get("mixing_index"),
            "edistance_global": data.get("edistance_global"),
            "edistance_local": data.get("edistance_local"),
            "edistance_pca": data.get("edistance_pca"),
            "edistance_pca_log": data.get("edistance_pca_log"),           
            "rmse": data.get("rmse"),
            "mse_lfc": data.get("mse_lfc"),
            "nb_deviance": data.get("nb_deviance"),
        })
    except Exception as e:
        print(f"Error processing file {fp}: {e}")
        continue

data_df = pd.DataFrame(rows)
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,nb_deviance
0,C57BL6J-2.036,baseline-cf,GABAergic neuron_Fiber_tracts,50,0.570374,0.608349,0.00,0.0,0.00,0.08,0.665761,36.035931,38.167051,1326.321997,23.548703,605.263245,373.183899,NaN
1,C57BL6J-2.036,baseline-cf,GABAergic neuron_Isocortex,50,0.670297,0.648808,0.16,1.0,0.16,0.60,0.675934,43.563454,47.745749,916.233893,24.244977,1496.381836,128.245697,NaN
2,C57BL6J-2.036,baseline-cf,astrocyte_Fiber_tracts,50,0.382818,0.415556,0.10,1.0,0.10,0.82,0.126393,32.461658,32.451422,1030.932532,13.411697,1876.262451,112.146637,NaN
3,C57BL6J-2.036,baseline-cf,astrocyte_Isocortex,50,0.735864,0.676738,0.24,1.0,0.24,0.86,0.009123,60.480022,62.360514,1787.439282,30.966541,3056.416260,113.270416,NaN
4,C57BL6J-2.036,baseline-cf,endothelial cell_Fiber_tracts,50,0.498319,0.576818,0.06,1.0,0.06,0.86,0.055016,36.975990,37.578715,1332.185315,17.214149,1701.265381,140.084076,NaN


In [6]:
# Remove -cf from the end of each model_name
data_df["model_name"] = data_df["model_name"].str.replace("-cf", "", regex=False)
n_deg = data_df["n_deg"].iloc[0]
data_df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,direction_match_gt,mixing_index,edistance_global,edistance_local,edistance_pca,edistance_pca_log,rmse,mse_lfc,nb_deviance
0,C57BL6J-2.036,baseline,GABAergic neuron_Fiber_tracts,50,0.570374,0.608349,0.00,0.0,0.00,0.08,0.665761,36.035931,38.167051,1326.321997,23.548703,605.263245,373.183899,NaN
1,C57BL6J-2.036,baseline,GABAergic neuron_Isocortex,50,0.670297,0.648808,0.16,1.0,0.16,0.60,0.675934,43.563454,47.745749,916.233893,24.244977,1496.381836,128.245697,NaN
2,C57BL6J-2.036,baseline,astrocyte_Fiber_tracts,50,0.382818,0.415556,0.10,1.0,0.10,0.82,0.126393,32.461658,32.451422,1030.932532,13.411697,1876.262451,112.146637,NaN
3,C57BL6J-2.036,baseline,astrocyte_Isocortex,50,0.735864,0.676738,0.24,1.0,0.24,0.86,0.009123,60.480022,62.360514,1787.439282,30.966541,3056.416260,113.270416,NaN
4,C57BL6J-2.036,baseline,endothelial cell_Fiber_tracts,50,0.498319,0.576818,0.06,1.0,0.06,0.86,0.055016,36.975990,37.578715,1332.185315,17.214149,1701.265381,140.084076,NaN


In [7]:
df = data_df.copy() # start with existing dataframe
df["sid"] = df["sid"].astype(str)

In [8]:
# Remove model_name 'cellina', 'cellina-ablated', 'cellina-graph'
df = df[~df["model_name"].isin(["cellina", "cellina-ablated", "cellina-graph", "cellina-W-1", "cellina-W-2", "cpa-1", "cpa-2", "scgen-1", "scgen-2", "baseline"])]
df.model_name.unique()

array(['cellina-W', 'cellina-ablated-W', 'cellina-graph-W', 'cpa',
       'scgen'], dtype=object)

In [9]:
# Remove -W from model names
df["model_name"] = df["model_name"].str.replace("-W", "", regex=False)
df.model_name.unique()

array(['cellina', 'cellina-ablated', 'cellina-graph', 'cpa', 'scgen'],
      dtype=object)

In [12]:
# CRC
# Print mean and std for each model_name over 'nb_deviance' column, rounded to 3 decimal places
df.groupby("model_name")["nb_deviance"].agg(["mean", "std"]).sort_values(by="mean", ascending=False).round(3)

,mean,std
model_name,,
scgen,0.087,0.035
cpa,0.042,0.047
cellina-ablated,0.036,0.043
cellina-graph,0.032,0.040
cellina,0.030,0.035


In [11]:
# MERFISH
df.groupby("model_name")["nb_deviance"].agg(["mean", "std"]).sort_values(by="mean", ascending=False).round(3)

,mean,std
model_name,,
scgen,0.328,0.054
cellina-ablated,0.088,0.109
cpa,0.053,0.040
cellina,0.042,0.042
cellina-graph,0.036,0.037
